# Scraping de reseñas — Hotel Olh Castellana Real (KAYAK)

Notebook para recuperar las reseñas publicadas en la ficha de KAYAK del
**Hotel Olh Castellana Real** (Cali, Colombia):

`https://www.kayak.com.co/Santiago-de-Cali-Hoteles-Castellana-Real.361893.ksp`

KAYAK renderiza la sección **Opiniones** con JavaScript y carga los textos vía
peticiones XHR internas. Por eso usamos **Playwright (Chromium headless)** y:

1. Cargamos la página con un User-Agent realista y esperamos a que se hidrate.
2. Hacemos scroll hasta la sección de opiniones e iteramos sobre el botón
   *“Más opiniones”* / paginación.
3. **Interceptamos las respuestas JSON** internas de KAYAK como fuente
   primaria (más limpio que parsear el DOM).
4. Como respaldo, parseamos el HTML final con BeautifulSoup.
5. Guardamos los resultados en `data/kayak_reviews_castellana_real.{csv,json}`.

> Aviso: el scraping debe respetar los Términos de Uso del sitio y la
> normativa local (Habeas Data en Colombia). Este notebook está pensado para
> uso **académico**, sin redistribuir los datos. Si necesitas un volumen
> grande y estable, valora usar la API oficial de Booking/Tripadvisor.

## 1. Instalación de dependencias

Si ya las tienes instaladas en `notebooks/.venv`, puedes saltarte estas
celdas. Solo necesitas ejecutar `playwright install chromium` una vez por
máquina para descargar el binario del navegador.

In [7]:
%pip install --quiet playwright beautifulsoup4 lxml pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


c:\Estudio\Maestria\Tesis\notebooks\.venv\Scripts\python.exe: No module named pip


In [8]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=False)

CompletedProcess(args=['c:\\Estudio\\Maestria\\Tesis\\notebooks\\.venv\\Scripts\\python.exe', '-m', 'playwright', 'install', 'chromium'], returncode=0)

## 2. Configuración

`HEADLESS=False` ayuda a depurar visualmente la primera vez (verás el
navegador). Una vez funcione, pásalo a `True`.

In [9]:
from __future__ import annotations

import asyncio
import json
import re
from dataclasses import dataclass, asdict, field
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
from bs4 import BeautifulSoup

URL = "https://www.kayak.com.co/Santiago-de-Cali-Hoteles-Castellana-Real.361893.ksp"
HOTEL_NOMBRE = "Hotel Olh Castellana Real"
HOTEL_CIUDAD = "Santiago de Cali"

HEADLESS = False
TIMEOUT_MS = 60_000
MAX_PAGINAS = 60
WAIT_BETWEEN_CLICKS_MS = 1_500

UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)
TS = datetime.now().strftime("%Y%m%d_%H%M%S")

OUT_HTML = OUT_DIR / f"kayak_castellana_real_{TS}.html"
OUT_XHR  = OUT_DIR / f"kayak_castellana_real_{TS}_xhr.jsonl"
OUT_CSV  = OUT_DIR / f"kayak_castellana_real_{TS}.csv"
OUT_JSON = OUT_DIR / f"kayak_castellana_real_{TS}.json"

print("Salida:", OUT_DIR.resolve())

Salida: C:\Estudio\Maestria\Tesis\notebooks\data


## 3. Modelo de datos

Estructura única para reviews extraídas. Si más adelante alimentamos la
tabla `reviews` de Neon, los campos clave son `texto`, `fecha_review`,
`puntuacion`, `plataforma` (`kayak/booking`), `idioma`.

In [10]:
@dataclass
class Review:
    autor: str | None = None
    pais: str | None = None
    fecha_review: str | None = None
    puntuacion: float | None = None
    titulo: str | None = None
    texto_positivo: str | None = None
    texto_negativo: str | None = None
    texto: str | None = None
    idioma: str | None = None
    tipo_viaje: str | None = None
    plataforma: str = "kayak"
    fuente_proveedor: str | None = None
    raw: dict[str, Any] = field(default_factory=dict)

    def to_row(self) -> dict[str, Any]:
        d = asdict(self)
        d.pop("raw", None)
        return d

## 4. Scraping con Playwright

Flujo:

1. Lanzamos Chromium con un User-Agent realista y locale `es-CO`.
2. Registramos un *listener* `page.on("response", ...)` para capturar **toda
   respuesta JSON** que contenga reviews. Las guardamos en
   `OUT_XHR` (formato JSONL) por si KAYAK expone un endpoint paginado.
3. Hacemos scroll hasta la sección **Opiniones** (anchor `#reviews-list`
   o el bloque que contenga "opiniones verificadas").
4. Pulsamos repetidamente el botón "Más opiniones" mientras esté visible
   y cambien los conteos, o avanzamos por paginador.
5. Cuando ya no haya cambios, guardamos el HTML final renderizado.

In [11]:
from playwright.sync_api import sync_playwright, Response, TimeoutError as PWTimeout

REVIEW_HINTS = ("review", "opinio", "guestreview", "userreview")


def scrape_kayak(url: str = URL, headless: bool = HEADLESS) -> dict[str, Any]:
    """Versión sync de Playwright (evita el NotImplementedError de
    asyncio.subprocess en Windows + SelectorEventLoop usado por Jupyter)."""

    xhr_payloads: list[dict[str, Any]] = []

    def handle_response(resp: Response) -> None:
        try:
            ct = (resp.headers or {}).get("content-type", "")
            if "application/json" not in ct:
                return
            u = resp.url.lower()
            if not any(h in u for h in REVIEW_HINTS):
                return
            data = resp.json()
            xhr_payloads.append({"url": resp.url, "status": resp.status, "data": data})
        except Exception:
            pass

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=headless)
        context = browser.new_context(
            user_agent=UA,
            locale="es-CO",
            viewport={"width": 1366, "height": 900},
            extra_http_headers={"Accept-Language": "es-CO,es;q=0.9,en;q=0.8"},
        )
        page = context.new_page()
        page.on("response", handle_response)

        page.goto(url, timeout=TIMEOUT_MS, wait_until="domcontentloaded")
        try:
            page.wait_for_load_state("networkidle", timeout=15_000)
        except PWTimeout:
            pass

        for sel in [
            'button:has-text("Aceptar")',
            'button:has-text("Acepto")',
            'button[aria-label*="acept" i]',
        ]:
            try:
                btn = page.locator(sel).first
                if btn.is_visible(timeout=1_500):
                    btn.click()
                    break
            except Exception:
                continue

        for sel in [
            'a:has-text("Opiniones")',
            'h2:has-text("Opiniones")',
            'section:has-text("opiniones verificadas")',
        ]:
            try:
                loc = page.locator(sel).first
                if loc.count():
                    loc.scroll_into_view_if_needed(timeout=3_000)
                    break
            except Exception:
                continue

        def count_reviews() -> int:
            return page.evaluate(
                """
                () => document.querySelectorAll(
                  '[data-testid*="review" i], [class*="review" i] article, li[class*="review" i]'
                ).length
                """
            )

        last_count = 0
        stable_rounds = 0
        for i in range(MAX_PAGINAS):
            for _ in range(6):
                page.mouse.wheel(0, 1400)
                page.wait_for_timeout(300)

            clicked = False
            for sel in [
                'button:has-text("Más opiniones")',
                'button:has-text("Ver más opiniones")',
                'button:has-text("Mostrar más")',
                'button:has-text("Cargar más")',
                'a:has-text("Siguiente")',
                'button[aria-label*="siguiente" i]',
            ]:
                try:
                    btn = page.locator(sel).first
                    if btn.is_visible(timeout=800):
                        btn.scroll_into_view_if_needed()
                        btn.click(timeout=2_000)
                        clicked = True
                        break
                except Exception:
                    continue

            page.wait_for_timeout(WAIT_BETWEEN_CLICKS_MS)
            cnt = count_reviews()
            print(f"[iter {i:02d}] reviews en DOM = {cnt}  (xhr capturados: {len(xhr_payloads)})")

            if cnt == last_count and not clicked:
                stable_rounds += 1
                if stable_rounds >= 2:
                    break
            else:
                stable_rounds = 0
            last_count = cnt

        html = page.content()
        OUT_HTML.write_text(html, encoding="utf-8")

        with OUT_XHR.open("w", encoding="utf-8") as f:
            for item in xhr_payloads:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        browser.close()

    return {"html_path": str(OUT_HTML), "xhr_path": str(OUT_XHR), "xhr_count": len(xhr_payloads)}

In [23]:
import threading, queue, sys, asyncio

def _run_in_thread(fn, *args, **kwargs):
    """Ejecuta `fn` en un hilo aparte y fuerza ProactorEventLoop en Windows.

    Contexto del bug:
    - En Jupyter, Tornado fija `WindowsSelectorEventLoopPolicy` como
      policy global del proceso.
    - Playwright sync internamente llama `asyncio.new_event_loop()`
      (no usa el loop "current"), por lo que la policy global decide
      qué clase de loop se construye.
    - Un `_WindowsSelectorEventLoop` no implementa `subprocess_exec`,
      así que al lanzar el driver Node de Playwright se obtiene
      `NotImplementedError` en Py 3.14 + Windows.

    Solución: en el hilo de trabajo monkey-patcheamos
    `asyncio.new_event_loop` para que devuelva un `ProactorEventLoop`
    mientras corre `fn`, y lo restauramos en `finally`. Es más local
    que cambiar la policy global y no pelea con Tornado.
    """
    q: queue.Queue = queue.Queue()

    def _worker():
        _orig_new_event_loop = asyncio.new_event_loop
        try:
            if sys.platform == "win32":
                asyncio.new_event_loop = lambda: asyncio.ProactorEventLoop()
                asyncio.set_event_loop(asyncio.ProactorEventLoop())
            q.put(("ok", fn(*args, **kwargs)))
        except BaseException as e:
            q.put(("err", e))
        finally:
            asyncio.new_event_loop = _orig_new_event_loop

    t = threading.Thread(target=_worker, daemon=True)
    t.start()
    t.join()
    kind, val = q.get()
    if kind == "err":
        raise val
    return val


result = _run_in_thread(scrape_kayak)
result

[iter 00] reviews en DOM = 0  (xhr capturados: 2)
[iter 01] reviews en DOM = 0  (xhr capturados: 3)
[iter 02] reviews en DOM = 0  (xhr capturados: 4)
[iter 03] reviews en DOM = 0  (xhr capturados: 5)
[iter 04] reviews en DOM = 0  (xhr capturados: 6)
[iter 05] reviews en DOM = 0  (xhr capturados: 7)
[iter 06] reviews en DOM = 0  (xhr capturados: 8)
[iter 07] reviews en DOM = 0  (xhr capturados: 9)
[iter 08] reviews en DOM = 0  (xhr capturados: 10)
[iter 09] reviews en DOM = 0  (xhr capturados: 11)
[iter 10] reviews en DOM = 0  (xhr capturados: 12)
[iter 11] reviews en DOM = 0  (xhr capturados: 13)
[iter 12] reviews en DOM = 0  (xhr capturados: 14)
[iter 13] reviews en DOM = 0  (xhr capturados: 15)
[iter 14] reviews en DOM = 0  (xhr capturados: 16)
[iter 15] reviews en DOM = 0  (xhr capturados: 17)
[iter 16] reviews en DOM = 0  (xhr capturados: 18)
[iter 17] reviews en DOM = 0  (xhr capturados: 19)
[iter 18] reviews en DOM = 0  (xhr capturados: 20)
[iter 19] reviews en DOM = 0  (xhr capt

{'html_path': 'data\\kayak_castellana_real_20260512_221702.html',
 'xhr_path': 'data\\kayak_castellana_real_20260512_221702_xhr.jsonl',
 'xhr_count': 32}

In [29]:
print(result)

{'html_path': 'data\\kayak_castellana_real_20260512_221702.html', 'xhr_path': 'data\\kayak_castellana_real_20260512_221702_xhr.jsonl', 'xhr_count': 32}


## 5. Extracción desde los XHR capturados (preferido)

Si KAYAK respondió con JSON conteniendo reviews, lo recorremos primero.
Buscamos heurísticamente claves típicas (`positive`, `negative`, `rating`,
`date`, `text`, `comment`, `country`, `traveler`) y normalizamos al
`dataclass` `Review`.

In [28]:
REVIEW_LIKE_KEYS = {
    "text", "comment", "body", "content",
    "positive", "negative", "pros", "cons",
    "rating", "score", "stars",
    "author", "user", "reviewer", "guest",
    "date", "createdAt", "stayDate",
}


def _looks_like_review(obj: dict) -> bool:
    keys = {k.lower() for k in obj.keys()}
    score = sum(1 for k in keys if k in REVIEW_LIKE_KEYS)
    has_text = any(
        isinstance(obj.get(k), str) and len(obj.get(k) or "") > 20
        for k in obj.keys()
        if k.lower() in {"text", "comment", "body", "content", "positive", "negative", "pros", "cons"}
    )
    return score >= 2 and has_text


def _walk(node: Any):
    if isinstance(node, dict):
        if _looks_like_review(node):
            yield node
        for v in node.values():
            yield from _walk(v)
    elif isinstance(node, list):
        for v in node:
            yield from _walk(v)


def _get(d: dict, *names: str) -> Any:
    low = {k.lower(): k for k in d.keys()}
    for n in names:
        if n.lower() in low:
            return d[low[n.lower()]]
    return None


def review_from_json(obj: dict) -> Review:
    pos = _get(obj, "positive", "pros", "liked")
    neg = _get(obj, "negative", "cons", "disliked")
    txt = _get(obj, "text", "comment", "body", "content")
    parts = [p for p in [txt, pos and f"👍 {pos}", neg and f"👎 {neg}"] if p]
    return Review(
        autor=_get(obj, "author", "userName", "reviewer", "guest", "user"),
        pais=_get(obj, "country", "userCountry", "nationality"),
        fecha_review=str(_get(obj, "date", "createdAt", "reviewDate", "stayDate") or "") or None,
        puntuacion=_safe_float(_get(obj, "rating", "score", "overallRating", "stars")),
        titulo=_get(obj, "title", "headline"),
        texto_positivo=pos,
        texto_negativo=neg,
        texto=" \n".join(str(x) for x in parts) if parts else None,
        idioma=_get(obj, "language", "lang"),
        tipo_viaje=_get(obj, "travelerType", "tripType"),
        plataforma="kayak",
        fuente_proveedor=_get(obj, "provider", "source"),
        raw=obj,
    )


def _safe_float(x: Any) -> float | None:
    if x is None:
        return None
    try:
        return float(str(x).replace(",", "."))
    except Exception:
        return None


reviews_xhr: list[Review] = []
if OUT_XHR.exists():
    with OUT_XHR.open("r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            for obj in _walk(item["data"]):
                reviews_xhr.append(review_from_json(obj))

print(f"Reviews extraídas de XHR: {len(reviews_xhr)}")
if reviews_xhr:
    sample = reviews_xhr[0]
    print({k: v for k, v in sample.to_row().items() if v})

Reviews extraídas de XHR: 0


## 6. Respaldo: parseo del HTML renderizado

Si los XHR no devolvieron nada (porque KAYAK pinta las reviews
directamente en el DOM), parseamos el HTML final con BeautifulSoup.
Los selectores son heurísticos y pueden necesitar ajuste si KAYAK
cambia su marcado.

In [25]:
def _pick_bs_parser() -> str:
    """Elige el mejor parser disponible en este kernel.

    Importante: no basta con `import lxml`. bs4 construye su
    `builder_registry` al importar el paquete; si bs4 se importó antes
    de que lxml fuera instalable, la entrada del builder de lxml nunca
    se registra aunque después instales lxml. Para reflejar ese estado
    real, consultamos directamente `bs4.builder.builder_registry`.

    Si el builder de lxml no está registrado, intentamos re-registrarlo
    importando explícitamente `bs4.builder._lxml`. Si tampoco eso
    funciona, caemos a `html.parser` (stdlib, siempre presente).
    """
    from bs4.builder import builder_registry
    if builder_registry.lookup("lxml") is not None:
        return "lxml"
    try:
        from bs4.builder._lxml import LXMLTreeBuilder
        builder_registry.register(LXMLTreeBuilder)
        if builder_registry.lookup("lxml") is not None:
            return "lxml"
    except Exception:
        pass
    return "html.parser"


def parse_html_reviews(html: str) -> list[Review]:
    soup = BeautifulSoup(html, _pick_bs_parser())
    out: list[Review] = []

    candidates = soup.select(
        '[data-testid*="review" i], article[class*="review" i], li[class*="review" i], div[class*="reviewCard" i]'
    )
    seen: set[str] = set()
    for node in candidates:
        text = node.get_text(" ", strip=True)
        if not text or len(text) < 25:
            continue
        key = text[:120]
        if key in seen:
            continue
        seen.add(key)

        score = None
        score_el = node.select_one('[class*="score" i], [class*="rating" i], [data-testid*="score" i]')
        if score_el:
            m = re.search(r"(\d+[.,]?\d*)", score_el.get_text(" ", strip=True))
            if m:
                score = _safe_float(m.group(1))

        author_el = node.select_one('[class*="author" i], [class*="user" i], [data-testid*="author" i]')
        date_el = node.select_one('[class*="date" i], time')

        pos_el = node.select_one('[class*="positive" i], [class*="liked" i], [class*="pros" i]')
        neg_el = node.select_one('[class*="negative" i], [class*="disliked" i], [class*="cons" i]')

        pos = pos_el.get_text(" ", strip=True) if pos_el else None
        neg = neg_el.get_text(" ", strip=True) if neg_el else None

        cuerpo = None
        for sel in ('[class*="content" i]', '[class*="body" i]', '[class*="text" i]', 'p'):
            el = node.select_one(sel)
            if el and len(el.get_text(strip=True)) > 30:
                cuerpo = el.get_text(" ", strip=True)
                break

        parts = [p for p in [cuerpo, pos and f"👍 {pos}", neg and f"👎 {neg}"] if p]
        out.append(Review(
            autor=author_el.get_text(" ", strip=True) if author_el else None,
            fecha_review=date_el.get_text(" ", strip=True) if date_el else None,
            puntuacion=score,
            texto_positivo=pos,
            texto_negativo=neg,
            texto=" \n".join(parts) if parts else text,
            plataforma="kayak",
        ))
    return out


reviews_html: list[Review] = []
if OUT_HTML.exists():
    reviews_html = parse_html_reviews(OUT_HTML.read_text(encoding="utf-8"))

print(f"Reviews extraídas del HTML: {len(reviews_html)}")
if reviews_html:
    print(reviews_html[0].to_row())

Reviews extraídas del HTML: 0


## 7. Consolidación y guardado

Preferimos XHR si encontramos al menos 1 review; si no, usamos el HTML.
Deduplicamos por `(autor, fecha_review, texto)` y exportamos a CSV/JSON.

In [26]:
reviews: list[Review] = reviews_xhr if reviews_xhr else reviews_html

dedup: dict[tuple, Review] = {}
for r in reviews:
    key = (r.autor or "", r.fecha_review or "", (r.texto or "")[:200])
    dedup.setdefault(key, r)
reviews = list(dedup.values())

df = pd.DataFrame([r.to_row() for r in reviews])
df.insert(0, "hotel", HOTEL_NOMBRE)
df.insert(1, "ciudad", HOTEL_CIUDAD)
df.insert(2, "scraped_at", datetime.now().isoformat(timespec="seconds"))

df.to_csv(OUT_CSV, index=False, encoding="utf-8")
OUT_JSON.write_text(
    json.dumps([r.to_row() for r in reviews], ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Total reviews: {len(df)}")
print(f"CSV : {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
df.head(10)

Total reviews: 0
CSV : data\kayak_castellana_real_20260512_221702.csv
JSON: data\kayak_castellana_real_20260512_221702.json


,hotel,ciudad,scraped_at


## 8. Diagnóstico rápido (si `len(df) == 0`)

Si el DataFrame queda vacío, ejecuta este bloque para inspeccionar los
XHR capturados y los selectores presentes en el HTML guardado. Eso te
dará pistas para ajustar `_looks_like_review` o los selectores de
`parse_html_reviews`.

In [27]:
if OUT_XHR.exists():
    urls = []
    with OUT_XHR.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                urls.append(json.loads(line)["url"])
            except Exception:
                pass
    print("XHR URLs capturadas que contienen 'review/opinio':")
    for u in sorted(set(urls)):
        print(" -", u)

if OUT_HTML.exists():
    soup = BeautifulSoup(OUT_HTML.read_text(encoding="utf-8"), _pick_bs_parser())
    candidatos = []
    for el in soup.find_all(True):
        attrs = " ".join([str(v) for v in el.attrs.values() if isinstance(v, str)])
        if re.search(r"review|opinio", attrs, re.I):
            candidatos.append((el.name, attrs[:120]))
    print(f"\nTags con 'review/opinio' en sus atributos: {len(candidatos)}")
    for tag, sample in candidatos[:25]:
        print(f"  <{tag}>  {sample}")

XHR URLs capturadas que contienen 'review/opinio':
 - https://www.kayak.com.co/i/api/seo/reviews/v3/filtered?travelerTypes=&months=&tagClusterName=&searchText=&reviewSources=BOOKING&sortType=recent&includeReviewLink=true&reviewType=hotel&objectId=361893&includeObjectId=false&startIndex=0&amount=10&excludeInternalLinks=true
 - https://www.kayak.com.co/i/api/seo/reviews/v3/filtered?travelerTypes=&months=&tagClusterName=&searchText=&reviewSources=BOOKING&sortType=recent&includeReviewLink=true&reviewType=hotel&objectId=361893&includeObjectId=false&startIndex=10&amount=10&excludeInternalLinks=true
 - https://www.kayak.com.co/i/api/seo/reviews/v3/filtered?travelerTypes=&months=&tagClusterName=&searchText=&reviewSources=BOOKING&sortType=recent&includeReviewLink=true&reviewType=hotel&objectId=361893&includeObjectId=false&startIndex=100&amount=10&excludeInternalLinks=true
 - https://www.kayak.com.co/i/api/seo/reviews/v3/filtered?travelerTypes=&months=&tagClusterName=&searchText=&reviewSources=B

## 9. Notas

- KAYAK suele reusar reseñas provenientes de **Booking.com** (en esta
  ficha aparecen *315 opiniones de Booking*). Si necesitas el corpus
  completo y estable de las 342 reviews, la fuente original más sencilla
  de scrapear es la propia ficha del hotel en Booking.com (HTML estático
  con paginación).
- Si KAYAK aplica Cloudflare/captcha en ejecuciones sucesivas: usa
  `HEADLESS=False`, espacia ejecuciones, o lanza con
  `playwright-stealth`.
- Para integrar al pipeline de la tesis, las columnas de `df` ya están
  alineadas con la tabla `reviews` de Neon (`texto`, `fecha_review`,
  `idioma`, `plataforma`). Falta sólo `hotel_id` y `archivo_id`.